In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

file_path = r"C:\Users\chinn\FDP_MLOPS\house_m1\data\House_Price_Prediction_Dataset.xlsx"

print("File exists:", os.path.exists(file_path))

df = pd.read_excel(file_path)

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

display(df.head())

File exists: True
Dataset loaded successfully!
Dataset Shape: (500, 8)


,Area_sqft,Bedrooms,Bathrooms,Property_Age_years,Location,Furnishing_Status,Parking_Spaces,Price_in_Lakhs
0,832,1,1,23,Vijayawada,Furnished,3,78.78
1,2613,4,4,23,Vijayawada,Semi-Furnished,2,110.92
2,2302,3,2,8,Hyderabad,Furnished,1,143.45
3,1741,5,5,30,Guntur,Semi-Furnished,3,72.91
4,1726,2,3,14,Vijayawada,Semi-Furnished,2,85.54


In [3]:
print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
display(df.describe())

print("\nColumn Names:")
print(df.columns.tolist())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Area_sqft           500 non-null    int64  
 1   Bedrooms            500 non-null    int64  
 2   Bathrooms           500 non-null    int64  
 3   Property_Age_years  500 non-null    int64  
 4   Location            500 non-null    object 
 5   Furnishing_Status   500 non-null    object 
 6   Parking_Spaces      500 non-null    int64  
 7   Price_in_Lakhs      500 non-null    float64
dtypes: float64(1), int64(5), object(2)
memory usage: 31.4+ KB

Missing Values:
Area_sqft             0
Bedrooms              0
Bathrooms             0
Property_Age_years    0
Location              0
Furnishing_Status     0
Parking_Spaces        0
Price_in_Lakhs        0
dtype: int64

Statistical Summary:


,Area_sqft,Bedrooms,Bathrooms,Property_Age_years,Parking_Spaces,Price_in_Lakhs
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,1897.860000,3.396000,3.272000,14.694000,1.508000,98.915140
std,743.711362,1.500562,1.402123,8.903196,1.150906,30.052409
min,613.000000,1.000000,1.000000,0.000000,0.000000,35.320000
25%,1281.250000,2.000000,2.000000,7.000000,0.000000,75.390000
50%,1885.500000,3.000000,3.000000,14.000000,2.000000,96.020000
75%,2550.000000,5.000000,5.000000,23.000000,3.000000,119.267500
max,3195.000000,6.000000,5.000000,30.000000,3.000000,179.630000



Column Names:
['Area_sqft', 'Bedrooms', 'Bathrooms', 'Property_Age_years', 'Location', 'Furnishing_Status', 'Parking_Spaces', 'Price_in_Lakhs']


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Separate input features and target
X = df.drop("Price_in_Lakhs", axis=1)
y = df["Price_in_Lakhs"]

# Identify numerical and categorical columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Create model pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
print("Model Training Completed!")
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("Root Mean Squared Error:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

Model Training Completed!
Mean Absolute Error: 7.628902
Root Mean Squared Error: 9.306863109813104
R2 Score: 0.8969779856415555


In [7]:
import mlflow
import mlflow.sklearn
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set MLflow tracking database
db_path = Path("../mlflow.db").resolve()
tracking_uri = f"sqlite:///{db_path.as_posix()}"

mlflow.set_tracking_uri(tracking_uri)

# Create or select experiment
mlflow.set_experiment("House_Price_Prediction")

# Start MLflow run
with mlflow.start_run(run_name="RandomForest_HousePrice") as run:

    # Log parameters
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("test_size", 0.2)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Log metrics
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2_Score", r2)

    # Save model using cloudpickle
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="house_price_model",
        serialization_format="cloudpickle"
    )

    print("MLflow Tracking Completed!")
    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2 Score:", r2)

2026/11/04 11:35:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MLflow Tracking Completed!
Run ID: bb6b98daf7bd4fd1b255070c445564bb
Model URI: models:/m-a53126b71281479cba0ed7a325c18d3c
MAE: 7.628902
RMSE: 9.306863109813104
R2 Score: 0.8969779856415555


In [8]:
from pathlib import Path

print("Notebook working folder:")
print(Path.cwd())

print("\nDatabase used by MLflow:")
print(Path("../mlflow.db").resolve())

Notebook working folder:
C:\Users\chinn\FDP_MLOPS\house_m1

Database used by MLflow:
C:\Users\chinn\FDP_MLOPS\mlflow.db
